In [0]:
-----------------------------------------QUERY PARA OBTENER CADA PRODUCTO CON SU INFORMACION DEL CATALOGO-----------------------------------------
WITH scraped_prods_with_catalog AS (
  SELECT
  catal.product_id,
  catal.brand,
  -- scrap.brand,
  scrap.sku,
  scrap.name,
  catal.retailer,
  catal.main_category,
  catal.sub_category,
  scrap.main_category,
  scrap.sub_category,
  TRY_CAST(REPLACE(REPLACE(scrap.list_price,'$',''),'.','') AS DOUBLE) AS list_price,
  TRY_CAST(REPLACE(REPLACE(scrap.cash_price,'$',''),'.','') AS DOUBLE) AS cash_price,
  TRY_CAST(scrap.scraped_at AS DATE),
  catal.link,
  scrap.installments_json
FROM
  products.bronze_scraped_products scrap
JOIN
  products.catalog catal
ON
  scrap.product_id = catal.product_id AND scrap.retailer = catal.retailer
)
SELECT
  *,
  CASE
    WHEN (sku IS NULL) OR ((list_price IS NULL)AND (cash_price IS NULL)) THEN FALSE
    ELSE TRUE
  END AS is_available,
  CASE
    WHEN name IS NULL OR name = '' THEN FALSE
    ELSE TRUE
  END AS is_valid_name,
  CASE
    WHEN list_price IS NULL AND cash_price IS NULL THEN FALSE
    WHEN list_price > 10000000 AND cash_price > 10000000 THEN FALSE
    WHEN list_price < 0 AND cash_price < 0 THEN FALSE
    ELSE TRUE
  END AS is_valid_price,
  CASE
    WHEN list_price IS NULL OR cash_price IS NULL OR list_price = 0 THEN "0"
    ELSE CONCAT(ROUND((1 - (cash_price / list_price)) * 100,2), "%")
  END AS discount_applied_str,
    CASE
    WHEN list_price IS NULL OR cash_price IS NULL OR list_price = 0 THEN 0.0
    ELSE ROUND((1 - (cash_price / list_price)) * 100, 2)
  END AS discount_applied_int,
  CASE
    -- Si el campo tiene contenido relevante o menciona cuotas sin interés
    WHEN (LOWER(installments_json) LIKE '%sin interés%' OR LOWER(installments_json) LIKE '%cuotas fijas%') THEN TRUE
    -- Si no tiene texto pero el número extraído es mayor a 1 (Caso Fravega/Megatone)
    WHEN ARRAY_MAX(
           TRANSFORM(
             REGEXP_EXTRACT_ALL(installments_json, ':\\s*"*(\\d+)', 1), 
             x -> CAST(x AS INT)
           )
         ) > 1 THEN TRUE
    ELSE FALSE
  END AS has_installments
FROM
  scraped_prods_with_catalog
ORDER BY
  scraped_at DESC;




SELECT COUNT(*) FROM products.silver_products;
SELECT * FROM products.silver_products ORDER BY scraped_at DESC;
SELECT * FROM products.bronze_scraped_products_clean ORDER BY scraped_at DESC LIMIT 10;



-----------------------------------------QUERY PARA OBTENER CADA PRODUCTO CON SU MAXIMA ACTUALIZACION DE PRECIOS Y STOCK-----------------------------------------
WITH max_scraped_at AS (
  SELECT
    catal.product_id,
    catal.brand,
    scrap.brand,
    scrap.sku,
    scrap.name,
    catal.retailer,
    catal.main_category,
    catal.sub_category,
    scrap.main_category,
    scrap.sub_category,
    TRY_CAST(NULLIF(REPLACE(REPLACE(scrap.list_price,'$',''),'.',''), '') AS DOUBLE) AS list_price,
    TRY_CAST(NULLIF(REPLACE(REPLACE(scrap.cash_price,'$',''),'.',''), '') AS DOUBLE) AS cash_price,
    CASE 
      WHEN (scrap.sku IS NULL OR scrap.sku = '') 
           OR ((scrap.list_price IS NULL OR scrap.list_price = '') 
               AND (scrap.cash_price IS NULL OR scrap.cash_price = ''))
      THEN "False"
      ELSE "True"
    END AS is_available,
    scrap.scraped_at AS updated_at,
    catal.link,
    ROW_NUMBER() OVER (PARTITION BY scrap.product_id, scrap.retailer ORDER BY scrap.scraped_at DESC) AS rn
  FROM products.bronze_scraped_products scrap
  JOIN products.catalog catal
    ON scrap.product_id = catal.product_id 
   AND scrap.retailer = catal.retailer
)
SELECT * FROM max_scraped_at WHERE rn = 1 ORDER BY updated_at DESC





SELECT 
  b.product_id,
  b.retailer,
  b.scraped_at,
  COUNT(*) as cnt
FROM workspace.products.bronze_scraped_products b
INNER JOIN workspace.products.catalog c 
  ON b.product_id = c.product_id 
 AND b.retailer = c.retailer
GROUP BY 1,2,3
HAVING COUNT(*) > 1